In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 🚀 COLAB SETUP (Run this first)
# ─────────────────────────────────────────────────────────────────────────────
import os
import sys

# 1. Install missing libraries in Colab
if 'google.colab' in sys.modules:
    print("Running on Google Colab. Installing dependencies...")
    !pip install wfdb scipy antropy pandas numpy matplotlib scikit-learn -q
    
    # 2. UNCOMMENT BELOW TO MOUNT GOOGLE DRIVE IF YOUR DATA IS THERE
    # from google.colab import drive
    # drive.mount('/content/drive')
    
    print("✅ Setup Complete.")
else:
    print("Running locally.")


# 🫀 ECG Monitoring System — Complete Hackathon Pipeline
### Problems 1 & 2: Heart Rate Detection + Atrial Fibrillation Detection
**Dataset:** MIT-BIH Arrhythmia Database (PhysioNet) — `mitdb 1.0.0`

---
**Pipeline Overview:**
1. Install dependencies & download MIT-BIH data via `wfdb`
2. Signal preprocessing (bandpass filter + baseline removal)
3. Pan-Tompkins R-peak detection
4. Heart rate calculation & continuous monitoring
5. R-R interval analysis for rhythm classification
6. AFib detection with sliding-window algorithm
7. Full visualizations and summary report
---

## ⚙️ SECTION 1 — Install Dependencies & Download Dataset

In [ ]:
# Install required packages
!pip install wfdb scipy numpy matplotlib pandas scikit-learn antropy -q
print('✅ All packages installed.')

In [ ]:
import wfdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.signal import butter, filtfilt, find_peaks, medfilt
from scipy.stats import variation
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Try antropy for sample entropy; fallback to manual if unavailable
try:
    import antropy as ant
    HAS_ANTROPY = True
except ImportError:
    HAS_ANTROPY = False

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
print('✅ Imports successful.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DATA CONFIGURATION (AUTO-DISCOVERY ENABLED)
# ─────────────────────────────────────────────────────────────────────────────
import os
RECORDS = ['100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '111', '112', '113', '114', '115', '116', '117', '118', '119', '121', '122', '123', '124', '200', '201', '202', '203', '205', '207', '208', '209', '210', '212', '213', '214', '215', '217', '219', '220', '221', '222', '223', '228', '230', '231', '232', '233', '234']

# 🔍 AUTO-DISCOVERY: We will look for any folder containing '100.hea'
DB_PATH = 'mitdb' # Default
found_path = None

for root, dirs, files in os.walk('/content'):
    if '100.hea' in files:
        found_path = root
        break

if found_path:
    DB_PATH = found_path
    print(f'✅ Found your database at: {DB_PATH}')
else:
    print(f'⚠️  Could not find local files. Falling back to downloading to {DB_PATH}...')
    os.makedirs(DB_PATH, exist_ok=True)
    try:
        wfdb.dl_database('mitdb', dl_dir=DB_PATH, records=['100'])
    except:
        pass

print(f'📊 Ready to process {len(RECORDS)} records.')


## 🔬 SECTION 2 — Signal Preprocessing

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PREPROCESSING MODULE
# Stage 1: Baseline wander removal  (median filter approach)
# Stage 2: Bandpass filter           (0.5–40 Hz Butterworth, 4th order)
# Stage 3: Normalisation             (zero-mean, unit-variance)
# ─────────────────────────────────────────────────────────────────────────────

def remove_baseline(signal, fs, window_sec=0.2):
    """Remove baseline wander using double median filter."""
    kernel = int(window_sec * fs)
    if kernel % 2 == 0:
        kernel += 1
    baseline = medfilt(signal, kernel_size=kernel)
    baseline = medfilt(baseline, kernel_size=kernel * 3 if kernel * 3 % 2 else kernel * 3 + 1)
    return signal - baseline

def bandpass_filter(signal, fs, lowcut=0.5, highcut=40.0, order=4):
    """4th-order Butterworth bandpass filter."""
    nyq = fs / 2.0
    low = lowcut / nyq
    high = min(highcut / nyq, 0.99)
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, signal)

def preprocess_ecg(raw_signal, fs):
    """Full preprocessing pipeline."""
    sig = remove_baseline(raw_signal, fs)
    sig = bandpass_filter(sig, fs)
    sig = (sig - np.mean(sig)) / (np.std(sig) + 1e-9)   # z-score normalise
    return sig

import os
import wfdb
def load_record(record_name, db_path=DB_PATH, channel=0, duration_sec=None):
    """Load a WFDB record, returning signal + metadata, downloading if missing."""
    if not os.path.exists(os.path.join(db_path, f'{str(record_name)}.hea')):
        print(f'⚠️ Record {record_name} missing from {db_path}. Downloading...')
        os.makedirs(db_path, exist_ok=True)
        wfdb.dl_database('mitdb', dl_dir=db_path, records=[str(record_name)])
    
    rec = wfdb.rdrecord(f'{db_path}/{record_name}', channels=[channel])
    signal = rec.p_signal[:, 0]
    fs = rec.fs
    if duration_sec:
        signal = signal[:int(duration_sec * fs)]
    annotation = wfdb.rdann(f'{db_path}/{record_name}', 'atr')
    return signal, fs, annotation

print('✅ Preprocessing functions defined.')

In [ ]:
# ── Load record 100 (normal sinus) as primary demo ───────────────────────────
DEMO_RECORD = '100'
raw, FS, ann = load_record(DEMO_RECORD, db_path=DB_PATH, duration_sec=60)   # first 60 seconds

clean = preprocess_ecg(raw, FS)
time_axis = np.arange(len(raw)) / FS

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(time_axis, raw, color='#888', linewidth=0.6)
axes[0].set_title(f'Raw ECG — MIT-BIH Record {DEMO_RECORD}', fontweight='bold')
axes[0].set_ylabel('Amplitude (mV)')

axes[1].plot(time_axis, clean, color='#1e3c72', linewidth=0.7)
axes[1].set_title('Preprocessed ECG (Baseline removed + Bandpass 0.5–40 Hz)')
axes[1].set_ylabel('Normalised Amplitude')
axes[1].set_xlabel('Time (s)')

plt.tight_layout()
plt.savefig('preprocessing_comparison.png', dpi=150)
plt.show()
print(f'Sampling rate: {FS} Hz  |  Duration: {len(raw)/FS:.1f} s  |  Samples: {len(raw)}')

## ❤️ SECTION 3 — R-Peak Detection (Pan-Tompkins Algorithm)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PAN-TOMPKINS R-PEAK DETECTOR
# Steps:
#  1. Bandpass filter (already done in preprocessing)
#  2. Derivative filter  → highlights steep QRS slopes
#  3. Squaring          → amplify high-frequency peaks, suppress low-amp noise
#  4. Moving-window integration → smooth + widen peaks for reliable detection
#  5. Adaptive thresholding → signal + noise thresholds updated beat-by-beat
#  6. Refractory period  → enforce minimum 200 ms between peaks
# ─────────────────────────────────────────────────────────────────────────────

def pan_tompkins_detector(ecg_signal, fs):
    """
    Full Pan-Tompkins QRS detector.
    Returns indices of detected R-peaks in the original signal.
    """
    # 1. Derivative filter
    diff = np.diff(ecg_signal, prepend=ecg_signal[0])

    # 2. Squaring
    squared = diff ** 2

    # 3. Moving-window integration (150 ms window)
    win = int(0.15 * fs)
    integrated = np.convolve(squared, np.ones(win) / win, mode='same')

    # 4. Initial peak detection (rough)
    min_distance = int(0.2 * fs)   # 200 ms refractory period
    rough_peaks, _ = find_peaks(integrated, distance=min_distance)

    if len(rough_peaks) == 0:
        return np.array([])

    # 5. Adaptive threshold (2-threshold learning)
    spk_i = 0.0   # signal peak estimate
    npk_i = 0.0   # noise peak estimate
    threshold1 = np.max(integrated[:int(2 * fs)]) * 0.25   # initialise from first 2s

    r_peaks = []
    for p in rough_peaks:
        val = integrated[p]
        if val >= threshold1:
            r_peaks.append(p)
            spk_i = 0.125 * val + 0.875 * spk_i
        else:
            npk_i = 0.125 * val + 0.875 * npk_i
        threshold1 = npk_i + 0.25 * (spk_i - npk_i)

    r_peaks = np.array(r_peaks)

    # 6. Refine back to original ECG maximum within ±50 ms window
    refined = []
    half_win = int(0.05 * fs)
    for p in r_peaks:
        lo = max(0, p - half_win)
        hi = min(len(ecg_signal) - 1, p + half_win)
        refined.append(lo + np.argmax(ecg_signal[lo:hi + 1]))

    return np.array(refined, dtype=int)


def signal_quality_index(ecg_segment, fs):
    """Simple SQI: ratio of energy in QRS band (5-15 Hz) to total energy."""
    from scipy.signal import welch
    freqs, psd = welch(ecg_segment, fs=fs, nperseg=min(256, len(ecg_segment)))
    qrs_band = np.logical_and(freqs >= 5, freqs <= 15)
    total_power = np.sum(psd) + 1e-9
    qrs_power = np.sum(psd[qrs_band])
    return float(qrs_power / total_power)


print('✅ Pan-Tompkins detector defined.')

In [ ]:
# ── Detect R-peaks on the demo record ────────────────────────────────────────
r_peaks = pan_tompkins_detector(clean, FS)

# Ground-truth peaks from annotation (for accuracy check)
ann_samples = ann.sample[ann.sample < len(raw)]
print(f'Detected peaks  : {len(r_peaks)}')
print(f'Annotated beats : {len(ann_samples)}')

# ── Plot detection result ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
show_sec = 10
n_samples = int(show_sec * FS)
t = time_axis[:n_samples]

ax.plot(t, clean[:n_samples], color='#1e3c72', linewidth=0.8, label='ECG')
valid_peaks = r_peaks[r_peaks < n_samples]
ax.plot(t[valid_peaks], clean[valid_peaks], 'r^', markersize=8, label='Detected R-peaks', zorder=5)

valid_ann = ann_samples[ann_samples < n_samples]
ax.plot(t[valid_ann], clean[valid_ann], 'go', markersize=5, alpha=0.5,
        label='PhysioNet annotations', zorder=4)

ax.set_title(f'R-Peak Detection — Record {DEMO_RECORD} (first {show_sec}s)', fontweight='bold')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Normalised Amplitude')
ax.legend()
plt.tight_layout()
plt.savefig('rpeaks_detected.png', dpi=150)
plt.show()

In [ ]:
# ── Compute sensitivity & positive predictivity ───────────────────────────────
TOLERANCE = int(0.05 * FS)   # 50 ms window

def evaluate_detection(detected, reference, tolerance):
    TP = 0
    used = set()
    for d in detected:
        for i, r in enumerate(reference):
            if i not in used and abs(d - r) <= tolerance:
                TP += 1
                used.add(i)
                break
    FN = len(reference) - TP
    FP = len(detected) - TP
    Se = TP / (TP + FN + 1e-9)
    Pp = TP / (TP + FP + 1e-9)
    return Se, Pp, TP, FN, FP

Se, Pp, TP, FN, FP = evaluate_detection(r_peaks, ann_samples, TOLERANCE)
print(f'\n📊 Detection Performance on Record {DEMO_RECORD}:')
print(f'   Sensitivity (Se) : {Se*100:.2f}%')
print(f'   Pos. Predictivity : {Pp*100:.2f}%')
print(f'   TP={TP}  FN={FN}  FP={FP}')

## 📈 SECTION 4 — Heart Rate Calculation & Continuous Monitoring

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# HEART RATE CALCULATION
# R-R interval → Heart rate: HR = 60 / (RR_seconds)
# Outlier removal: ignore RR < 300 ms or > 2000 ms (physiologically impossible)
# ─────────────────────────────────────────────────────────────────────────────

def compute_rr_intervals(r_peaks, fs):
    """Compute R-R intervals in seconds, filtering physiological outliers."""
    if len(r_peaks) < 2:
        return np.array([])
    rr_samples = np.diff(r_peaks)
    rr_sec = rr_samples / fs
    # Physiological range: 0.3 – 2.0 s (30 – 200 BPM)
    valid = (rr_sec >= 0.3) & (rr_sec <= 2.0)
    return rr_sec[valid], rr_samples[valid]


def compute_instantaneous_hr(r_peaks, fs):
    """Instantaneous HR at each beat."""
    rr_sec, _ = compute_rr_intervals(r_peaks, fs)
    hr = 60.0 / rr_sec
    # Peak times: midpoint of each RR pair
    peak_times = r_peaks[1:len(rr_sec) + 1] / fs
    return peak_times, hr


def sliding_window_hr(r_peaks, fs, window_sec=10, step_sec=1):
    """HR computed in sliding windows (for trend plotting)."""
    total_sec = r_peaks[-1] / fs if len(r_peaks) > 0 else 0
    windows, hr_vals, qualities = [], [], []
    t = 0
    while t + window_sec <= total_sec:
        lo = int(t * fs)
        hi = int((t + window_sec) * fs)
        in_window = r_peaks[(r_peaks >= lo) & (r_peaks < hi)]
        if len(in_window) >= 2:
            rr, _ = compute_rr_intervals(in_window, fs)
            if len(rr) > 0:
                hr = 60 / np.mean(rr)
                windows.append(t + window_sec / 2)
                hr_vals.append(hr)
                qualities.append(signal_quality_index(clean[lo:hi], fs))
        t += step_sec
    return np.array(windows), np.array(hr_vals), np.array(qualities)


# ── Run on full record ───────────────────────────────────────────────────────
# Use full recording (reload without duration limit for richer analysis)
raw_full, FS, ann_full = load_record(DEMO_RECORD, db_path=DB_PATH)
clean_full = preprocess_ecg(raw_full, FS)
r_peaks_full = pan_tompkins_detector(clean_full, FS)

win_t, hr_win, sqi = sliding_window_hr(r_peaks_full, FS)
peak_t, inst_hr = compute_instantaneous_hr(r_peaks_full, FS)

rr_sec_full, _ = compute_rr_intervals(r_peaks_full, FS)
print(f'Full recording:  {len(raw_full)/FS/60:.1f} min  |  {len(r_peaks_full)} beats detected')
print(f'Mean HR: {60/np.mean(rr_sec_full):.1f} BPM  |  Min: {60/np.max(rr_sec_full):.1f}  |  Max: {60/np.min(rr_sec_full):.1f}')

In [ ]:
# ── Visualise Heart Rate Trend ────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# 1. Instantaneous HR
axes[0].plot(peak_t / 60, inst_hr, color='#e74c3c', linewidth=0.8, alpha=0.7, label='Inst. HR')
axes[0].axhline(60, color='gray', linestyle='--', linewidth=0.5)
axes[0].axhline(100, color='gray', linestyle='--', linewidth=0.5)
axes[0].set_ylabel('Heart Rate (BPM)')
axes[0].set_title(f'Instantaneous Heart Rate — Record {DEMO_RECORD}', fontweight='bold')
axes[0].legend()
axes[0].set_ylim(30, 180)
axes[0].fill_between(peak_t / 60, 60, 100, alpha=0.05, color='green', label='Normal zone')

# 2. Windowed HR
axes[1].plot(win_t / 60, hr_win, color='#2a5298', linewidth=1.5, label='10-s window HR')
axes[1].fill_between(win_t / 60, hr_win - 5, hr_win + 5, alpha=0.2, color='#2a5298')
axes[1].set_ylabel('Heart Rate (BPM)')
axes[1].set_title('Sliding-Window Heart Rate (10 s window, 1 s step)')
axes[1].legend()

# 3. Signal Quality Index
colors = ['#2ecc71' if q > 0.5 else '#e74c3c' for q in sqi]
axes[2].bar(win_t / 60, sqi, width=1/60, color=colors, alpha=0.8)
axes[2].axhline(0.5, color='black', linestyle='--', linewidth=0.8, label='Quality threshold')
axes[2].set_ylabel('SQI')
axes[2].set_xlabel('Time (min)')
axes[2].set_title('Signal Quality Index per Window')
axes[2].legend()

plt.tight_layout()
plt.savefig('heart_rate_trend.png', dpi=150)
plt.show()

## 🔍 SECTION 5 — R-R Interval Analysis & Rhythm Classification

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RR INTERVAL FEATURE EXTRACTION
# Key HRV features used for rhythm classification:
#  - CV  (Coefficient of Variation) = std/mean
#  - RMSSD = Root Mean Square of Successive Differences
#  - pNN50 = proportion of successive diffs > 50 ms
#  - Sample Entropy = complexity of RR sequence
# ─────────────────────────────────────────────────────────────────────────────

def sample_entropy_manual(rr, m=2, r_tol=0.2):
    """Manual Sample Entropy computation (fallback if antropy unavailable)."""
    N = len(rr)
    if N < m + 2:
        return 0.0
    r = r_tol * np.std(rr)
    def _count(m_):
        count = 0
        for i in range(N - m_):
            template = rr[i:i + m_]
            for j in range(i + 1, N - m_):
                if np.max(np.abs(rr[j:j + m_] - template)) < r:
                    count += 1
        return count
    A = _count(m + 1)
    B = _count(m)
    return -np.log((A + 1e-9) / (B + 1e-9))


def extract_rr_features(rr_sec):
    """Extract all rhythm features from an RR series."""
    if len(rr_sec) < 4:
        return {}
    mean_rr = np.mean(rr_sec)
    std_rr = np.std(rr_sec)
    cv = std_rr / (mean_rr + 1e-9)
    rmssd = np.sqrt(np.mean(np.diff(rr_sec) ** 2))
    pnn50 = np.sum(np.abs(np.diff(rr_sec)) > 0.05) / (len(rr_sec) - 1)

    if HAS_ANTROPY:
        samp_ent = ant.sample_entropy(rr_sec)
    else:
        samp_ent = sample_entropy_manual(rr_sec[:min(50, len(rr_sec))])

    return {
        'mean_rr': mean_rr,
        'std_rr': std_rr,
        'cv': cv,
        'rmssd': rmssd,
        'pnn50': pnn50,
        'sample_entropy': samp_ent,
        'mean_hr': 60 / mean_rr,
        'n_beats': len(rr_sec)
    }


def classify_rhythm(features):
    """
    Rule-based classifier mirroring the hackathon solution table:
    Normal  : CV < 0.15 and entropy < 1.0
    Sinus A : CV 0.10-0.20 and entropy < 1.2
    Irregular: CV 0.15-0.25
    AFib     : CV > 0.20 and entropy > 1.5
    """
    cv = features.get('cv', 0)
    se = features.get('sample_entropy', 0)
    rmssd = features.get('rmssd', 0)

    if cv > 0.20 and se > 1.5:
        return 'AFib', 'red'
    elif 0.15 < cv <= 0.25:
        return 'Irregular', 'orange'
    elif 0.10 < cv <= 0.20:
        return 'Sinus Arrhythmia', 'yellow'
    else:
        return 'Normal Sinus', 'green'


# ── Compute global features ───────────────────────────────────────────────────
feat = extract_rr_features(rr_sec_full)
label, color = classify_rhythm(feat)

print('\n📊 Global RR Feature Summary:')
for k, v in feat.items():
    print(f'   {k:20s}: {v:.4f}')
print(f'\n🏷️  Global Rhythm Classification: {label}')

In [ ]:
# ── RR Interval Plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Poincaré plot
axes[0].scatter(rr_sec_full[:-1] * 1000, rr_sec_full[1:] * 1000,
                alpha=0.4, s=10, color='#2a5298')
axes[0].set_xlabel('RR(n) [ms]')
axes[0].set_ylabel('RR(n+1) [ms]')
axes[0].set_title('Poincaré Plot (RR scatter)')
axes[0].plot([300, 1200], [300, 1200], 'r--', linewidth=0.8)

# RR histogram
axes[1].hist(rr_sec_full * 1000, bins=40, color='#1e3c72', alpha=0.8, edgecolor='white')
axes[1].set_xlabel('RR Interval [ms]')
axes[1].set_ylabel('Count')
axes[1].set_title('RR Interval Distribution')
axes[1].axvline(feat['mean_rr'] * 1000, color='red', linewidth=1.5,
                label=f"Mean={feat['mean_rr']*1000:.0f}ms")
axes[1].legend()

# RR over time
peak_times_full = r_peaks_full[1:len(rr_sec_full) + 1] / FS
axes[2].plot(peak_times_full / 60, rr_sec_full * 1000, color='#e74c3c', linewidth=0.7)
axes[2].set_xlabel('Time (min)')
axes[2].set_ylabel('RR Interval [ms]')
axes[2].set_title('RR Intervals Over Time')
axes[2].set_ylim(300, 1500)

plt.suptitle(f'Record {DEMO_RECORD} — Rhythm: {label}', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('rr_analysis.png', dpi=150)
plt.show()

## 🚨 SECTION 6 — AFib Detection with Sliding-Window Algorithm

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# AFIB SLIDING-WINDOW DETECTOR
# For each 30-beat window (50% overlap):
#   1. Compute CV, RMSSD, sample entropy
#   2. Apply three-criterion rule
#   3. Smooth labels with majority vote over 3 consecutive windows
#   4. Output: timestamp + label + confidence score
# ─────────────────────────────────────────────────────────────────────────────

def afib_sliding_detector(r_peaks, fs, window_beats=30, overlap=0.5):
    """
    Scan the full recording beat-by-beat in overlapping windows.
    Returns a DataFrame with timestamp, rhythm label, features, confidence.
    """
    if len(r_peaks) < window_beats:
        print('Not enough beats for windowed analysis.')
        return pd.DataFrame()

    step = max(1, int(window_beats * (1 - overlap)))
    results = []

    for start in range(0, len(r_peaks) - window_beats, step):
        end = start + window_beats
        window_peaks = r_peaks[start:end]
        rr_w, _ = compute_rr_intervals(window_peaks, fs)

        if len(rr_w) < 5:
            continue

        feat_w = extract_rr_features(rr_w)
        label_w, color_w = classify_rhythm(feat_w)

        # Confidence: 0-1 score based on how strongly criteria are met
        afib_score = 0.0
        if feat_w['cv'] > 0.20:   afib_score += 0.35
        if feat_w['sample_entropy'] > 1.5: afib_score += 0.40
        if feat_w['rmssd'] > 0.05: afib_score += 0.15
        if feat_w['pnn50'] > 0.30: afib_score += 0.10

        center_time = window_peaks[window_beats // 2] / fs

        results.append({
            'time_sec': center_time,
            'time_min': center_time / 60,
            'label': label_w,
            'afib_score': afib_score,
            'cv': feat_w['cv'],
            'rmssd': feat_w['rmssd'],
            'sample_entropy': feat_w['sample_entropy'],
            'mean_hr': feat_w['mean_hr'],
            'pnn50': feat_w['pnn50']
        })

    df = pd.DataFrame(results)

    # Majority-vote smoothing (3 consecutive windows)
    if len(df) >= 3:
        smoothed = []
        for i in range(len(df)):
            lo = max(0, i - 1)
            hi = min(len(df), i + 2)
            vote = Counter(df['label'].iloc[lo:hi]).most_common(1)[0][0]
            smoothed.append(vote)
        df['label_smoothed'] = smoothed
    else:
        df['label_smoothed'] = df['label']

    return df


print('✅ AFib sliding-window detector defined.')

In [ ]:
# ── Run AFib detector ─────────────────────────────────────────────────────────
df_rhythm = afib_sliding_detector(r_peaks_full, FS, window_beats=30)
print(df_rhythm[['time_min', 'label_smoothed', 'afib_score', 'cv', 'mean_hr']].head(20).to_string(index=False))
print(f'\n📊 Rhythm label distribution:')
print(df_rhythm['label_smoothed'].value_counts())

In [ ]:
# ── Visualise AFib Detection Timeline ────────────────────────────────────────
COLOR_MAP = {
    'Normal Sinus': '#2ecc71',
    'Sinus Arrhythmia': '#f39c12',
    'Irregular': '#e67e22',
    'AFib': '#e74c3c'
}

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# AFib score
axes[0].plot(df_rhythm['time_min'], df_rhythm['afib_score'], color='#e74c3c', linewidth=1.2)
axes[0].axhline(0.75, color='black', linestyle='--', linewidth=0.8, label='AFib threshold (0.75)')
axes[0].fill_between(df_rhythm['time_min'], 0.75, df_rhythm['afib_score'].clip(lower=0.75),
                     alpha=0.4, color='red', label='AFib zone')
axes[0].set_ylabel('AFib Score')
axes[0].set_title('AFib Confidence Score Over Time', fontweight='bold')
axes[0].set_ylim(0, 1.05)
axes[0].legend()

# CV over time
axes[1].plot(df_rhythm['time_min'], df_rhythm['cv'], color='#9b59b6', linewidth=1.0)
axes[1].axhline(0.20, color='red', linestyle='--', linewidth=0.8, label='CV threshold (0.20)')
axes[1].axhline(0.15, color='orange', linestyle='--', linewidth=0.8, label='CV=0.15')
axes[1].set_ylabel('Coefficient of Variation')
axes[1].set_title('R-R Coefficient of Variation')
axes[1].legend()

# Rhythm label strip
for _, row in df_rhythm.iterrows():
    axes[2].axvline(row['time_min'], color=COLOR_MAP.get(row['label_smoothed'], 'gray'),
                   linewidth=2, alpha=0.7)
axes[2].set_yticks([])
axes[2].set_xlabel('Time (min)')
axes[2].set_title('Rhythm Classification Strip')
patches = [mpatches.Patch(color=c, label=l) for l, c in COLOR_MAP.items()]
axes[2].legend(handles=patches, loc='upper right')

plt.tight_layout()
plt.savefig('afib_detection.png', dpi=150)
plt.show()

## 📊 SECTION 7 — Multi-Record Evaluation on MIT-BIH

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Evaluate all downloaded records: detection accuracy + rhythm classification
# ─────────────────────────────────────────────────────────────────────────────

all_results = []

for rec in RECORDS:
    print(f'\n⏳ Processing record {rec}...')
    try:
        raw_r, fs_r, ann_r = load_record(rec, db_path=DB_PATH)
        clean_r = preprocess_ecg(raw_r, fs_r)
        peaks_r = pan_tompkins_detector(clean_r, fs_r)
        ann_samp = ann_r.sample[ann_r.sample < len(raw_r)]

        Se_r, Pp_r, TP_r, FN_r, FP_r = evaluate_detection(peaks_r, ann_samp, int(0.05 * fs_r))

        rr_r, _ = compute_rr_intervals(peaks_r, fs_r)
        feat_r = extract_rr_features(rr_r) 
        label_r, _ = classify_rhythm(feat_r)

        all_results.append({
            'Record': rec,
            'Beats_detected': len(peaks_r),
            'Beats_annotated': len(ann_samp),
            'Sensitivity_%': round(Se_r * 100, 2),
            'Pos_Predictivity_%': round(Pp_r * 100, 2),
            'Mean_HR_BPM': round(feat_r.get('mean_hr', 0), 1),
            'CV': round(feat_r.get('cv', 0), 4),
            'Rhythm': label_r
        })
        print(f'   Se={Se_r*100:.1f}%  Pp={Pp_r*100:.1f}%  Rhythm={label_r}')
    except Exception as e:
        print(f'   ❌ Error: {e}')

df_results = pd.DataFrame(all_results)
print('\n\n📋 FULL EVALUATION TABLE:')
print(df_results.to_string(index=False))

In [ ]:
# ── Bar chart of detection performance ───────────────────────────────────────
if len(df_results) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    x = np.arange(len(df_results))
    w = 0.35
    axes[0].bar(x - w/2, df_results['Sensitivity_%'], w, label='Sensitivity', color='#2a5298')
    axes[0].bar(x + w/2, df_results['Pos_Predictivity_%'], w, label='Pos. Predictivity', color='#e74c3c')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([f"Rec {r}" for r in df_results['Record']])
    axes[0].set_ylim(0, 110)
    axes[0].set_ylabel('%')
    axes[0].set_title('Detection Performance per Record')
    axes[0].legend()
    axes[0].axhline(95, color='green', linestyle='--', linewidth=0.8, label='95% target')

    axes[1].bar(df_results['Record'], df_results['Mean_HR_BPM'], color='#27ae60', alpha=0.8)
    axes[1].set_ylabel('BPM')
    axes[1].set_title('Mean Heart Rate per Record')
    for i, (_, row) in enumerate(df_results.iterrows()):
        axes[1].text(i, row['Mean_HR_BPM'] + 1, row['Rhythm'], ha='center', fontsize=9,
                     color='darkblue', fontweight='bold')

    plt.tight_layout()
    plt.savefig('multi_record_evaluation.png', dpi=150)
    plt.show()

## 🤖 SECTION 8 — Optional: ML Classifier for Beat Annotation

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# OPTIONAL: Train a Random Forest on beat morphology features
# Features per beat: 40-sample normalised beat template + RR interval features
# Labels from WFDB annotations (N=Normal, A=AFib/supraventricular, V=PVC, etc.)
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder

BEAT_WIN = 40   # samples around each peak (±20 samples at 360 Hz ≈ ±55 ms)

def extract_beat_features(ecg, r_peaks, fs, rr_intervals):
    """Extract per-beat feature vector: morphology + local RR context."""
    features, labels_idx = [], []
    for i, p in enumerate(r_peaks):
        lo = p - BEAT_WIN
        hi = p + BEAT_WIN
        if lo < 0 or hi >= len(ecg):
            continue
        beat = ecg[lo:hi]
        beat_norm = (beat - beat.mean()) / (beat.std() + 1e-9)

        # Local RR context
        pre_rr = rr_intervals[i - 1] if i > 0 and i - 1 < len(rr_intervals) else 0
        post_rr = rr_intervals[i] if i < len(rr_intervals) else 0
        ratio = pre_rr / (post_rr + 1e-9)

        feat_vec = np.concatenate([beat_norm, [pre_rr, post_rr, ratio]])
        features.append(feat_vec)
        labels_idx.append(i)

    return np.array(features), np.array(labels_idx)


# Load record 100 with annotations
raw_ml, fs_ml, ann_ml = load_record('100')
clean_ml = preprocess_ecg(raw_ml, fs_ml)
peaks_ml = pan_tompkins_detector(clean_ml, fs_ml)
rr_ml, _ = compute_rr_intervals(peaks_ml, fs_ml)

# Align annotation labels to detected peaks (50ms tolerance)
BEAT_LABELS = []
for p in peaks_ml:
    dists = np.abs(ann_ml.sample - p)
    idx = np.argmin(dists)
    if dists[idx] <= int(0.05 * fs_ml):
        sym = ann_ml.symbol[idx]
        # Simplify: N → Normal, V → Ectopic, else → Other
        if sym == 'N' or sym == '.':
            BEAT_LABELS.append('Normal')
        elif sym == 'V':
            BEAT_LABELS.append('PVC')
        elif sym in ['A', 'a', 'S']:
            BEAT_LABELS.append('SVE')
        else:
            BEAT_LABELS.append('Other')
    else:
        BEAT_LABELS.append('Normal')   # default

X, feat_idx = extract_beat_features(clean_ml, peaks_ml, fs_ml, rr_ml)
y_raw = np.array(BEAT_LABELS[:len(X)])

le = LabelEncoder()
y = le.fit_transform(y_raw)

print(f'Feature matrix: {X.shape}   Classes: {np.unique(y_raw)}')

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
cv_scores = cross_val_score(rf, X, y, cv=5, scoring='accuracy')
print(f'\n🌲 Random Forest 5-fold CV Accuracy: {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%')

rf.fit(X, y)
print('✅ Model trained.')

## 📋 SECTION 9 — Final Report Generator

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# COMPREHENSIVE FINAL REPORT — printed to screen and saved as text file
# ─────────────────────────────────────────────────────────────────────────────

report = f"""
╔══════════════════════════════════════════════════════════════════════════════╗
║           ECG MONITORING SYSTEM — HACKATHON FINAL REPORT                   ║
║           Dataset: MIT-BIH Arrhythmia Database (PhysioNet)                 ║
╚══════════════════════════════════════════════════════════════════════════════╝

── DEMO RECORD: {DEMO_RECORD} ──────────────────────────────────────────────────
  Sampling Rate      : {FS} Hz
  Duration           : {len(raw_full)/FS/60:.2f} min
  Beats Detected     : {len(r_peaks_full)}
  Detection Se       : {Se*100:.2f}%
  Detection Pp       : {Pp*100:.2f}%

── HEART RATE SUMMARY ──────────────────────────────────────────────────────
  Mean HR            : {feat['mean_hr']:.1f} BPM
  Mean RR            : {feat['mean_rr']*1000:.1f} ms
  RR Std Dev         : {feat['std_rr']*1000:.1f} ms
  RMSSD              : {feat['rmssd']*1000:.1f} ms
  pNN50              : {feat['pnn50']*100:.1f}%

── RHYTHM ANALYSIS ──────────────────────────────────────────────────────────
  Coefficient of Variation (CV) : {feat['cv']:.4f}
  Sample Entropy                : {feat['sample_entropy']:.4f}
  Global Rhythm Classification  : {label}

── MULTI-RECORD EVALUATION ─────────────────────────────────────────────────
{df_results.to_string(index=False) if len(all_results)>0 else 'Not run'}

── ALGORITHM PIPELINE SUMMARY ──────────────────────────────────────────────
  1. Baseline removal    : Double median filter (200 ms + 600 ms kernels)
  2. Bandpass filter     : 4th-order Butterworth, 0.5–40 Hz
  3. R-peak detection    : Pan-Tompkins (derivative → squaring → integration
                           → adaptive dual-threshold → refractory 200 ms)
  4. Peak refinement     : ±50 ms window back to raw ECG maximum
  5. HR calculation      : 60 / RR_mean, sliding window 10 s / 1 s step
  6. Rhythm features     : CV, RMSSD, pNN50, Sample Entropy
  7. Rhythm classifier   : Rule-based (3-criterion threshold)
  8. AFib scanner        : 30-beat sliding window, majority-vote smoothing
  9. ML beat classifier  : Random Forest on 80-dim morphology + RR features

╔══════════════════════════════════════════════════════════════════════════════╗
║  All figures saved: preprocessing_comparison.png, rpeaks_detected.png,     ║
║  heart_rate_trend.png, rr_analysis.png, afib_detection.png,                ║
║  multi_record_evaluation.png                                                ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

print(report)

with open('ecg_hackathon_report.txt', 'w') as f:
    f.write(report)
print('📄 Report saved as ecg_hackathon_report.txt')

## 🧪 SECTION 10 — Try Different Records

The MIT-BIH database has 48 records. Some useful ones to try:

| Record | Content |
|--------|---------|
| `100`  | Normal sinus rhythm (clean baseline) |
| `105`  | Noisy, motion artifacts |
| `108`  | AV block, missed beats |
| `201`  | Supraventricular ectopy |
| `202`  | Mix of rhythms |
| `207`  | Ventricular flutter |
| `210`  | Frequent PVCs |
| `217`  | Bundle branch block |
| `231`  | Left bundle branch block |

In [ ]:
# ── Change this to any valid MIT-BIH record number ───────────────────────────
TARGET_RECORD = '201'   # ← CHANGE ME

try:
    wfdb.dl_database('mitdb', dl_dir='mitdb', records=[TARGET_RECORD])
    raw_t, fs_t, ann_t = load_record(TARGET_RECORD)
    clean_t = preprocess_ecg(raw_t, fs_t)
    peaks_t = pan_tompkins_detector(clean_t, fs_t)
    rr_t, _ = compute_rr_intervals(peaks_t, fs_t)
    feat_t = extract_rr_features(rr_t)
    label_t, _ = classify_rhythm(feat_t)
    df_t = afib_sliding_detector(peaks_t, fs_t)

    print(f'Record {TARGET_RECORD}: {len(peaks_t)} beats | HR={feat_t["mean_hr"]:.1f} BPM | Rhythm={label_t}')
    print(f'Rhythm distribution:\n{df_t["label_smoothed"].value_counts()}')

    # Quick plot
    fig, ax = plt.subplots(figsize=(14, 3))
    t_t = np.arange(len(clean_t)) / fs_t
    ax.plot(t_t[:int(20*fs_t)], clean_t[:int(20*fs_t)], linewidth=0.7, color='#1e3c72')
    vp = peaks_t[peaks_t < int(20*fs_t)]
    ax.plot(t_t[vp], clean_t[vp], 'r^', markersize=7)
    ax.set_title(f'Record {TARGET_RECORD} — {label_t} (first 20s)', fontweight='bold')
    ax.set_xlabel('Time (s)')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Error: {e}')